# Replay one latency spike

Compare two wake policies on the same synthetic trace, then inspect the retained observations around the spike.

In [ ]:
import os
os.environ["CC"] = "cc"
%pip -q install "git+https://github.com/pgg898200-ship-it/chronosieve.git@743092a8c0da995a2e5007d7fea56ae16e669fe2"

## Run the policies

The prediction stays at 12 ms and only tick 50 changes. When `should_wake` is true, the result also contains a Markdown digest.

In [ ]:
from chronosieve import ChronoSieve

latency = [12.0] * 50 + [31.0] + [12.0] * 12
for burst in (1, 2):
    channel = ChronoSieve(channel="latency_ms", anomaly_burst=burst)
    wakes = []
    for t, value in enumerate(latency):
        result = channel.step(t, value, preds=[12.0])
        if result["should_wake"]:
            wakes.append(t)
    samples = channel.zoom(49, 51)["records"]
    print(burst, wakes, [row["y"] for row in samples])

With a one-observation trigger, tick 50 produces a digest. The two-observation trigger stays quiet. Both return 12, 31, 12 ms for ticks 49 through 51.

In [ ]:
samples[1]

The raw record contains the actual reading, prediction and flags for tick 50.

[Pipeline API](https://github.com/pgg898200-ship-it/chronosieve/blob/main/python/chronosieve/pipeline.py) · [Recorded workloads](https://github.com/pgg898200-ship-it/chronosieve/blob/main/benchmarks/results_m4.json)